In [5]:
import pandas as pd
import sqlite3
import os

# -----------------------------
# Create folders if not exist
# -----------------------------
os.makedirs('raw_data', exist_ok=True)
os.makedirs('processed_data', exist_ok=True)

# -----------------------------
# Step 1: Data Lake (Raw Layer)
# -----------------------------
def load_to_data_lake():
    print("Loading data into Data Lake...")
    
    df = pd.read_csv('../raw_data/faculty.csv')
    
    # Store as raw copy (lake layer)
    df.to_csv('../raw_data/lake_faculty.csv', index=False)
    
    print("Data stored in Data Lake")

# -----------------------------
# Step 2: Processing Layer
# -----------------------------
def process_data():
    print("Processing data...")
    
    df = pd.read_csv('../raw_data/lake_faculty.csv')
    
    # ----------Cleaning------------------
    # Remove rows with missing important fields
    df = df.dropna(subset=['faculty_id', 'faculty_name', 'hours_assigned'])

    # Remove duplicates
    df = df.drop_duplicates()

    # Fix invalid hours (negative values)
    df = df[df['hours_assigned'] >= 0]

    # Convert datatype
    df['hours_assigned'] = df['hours_assigned'].astype(int)

    # Save processed data
    df.to_csv('../processed_data/processed_faculty.csv', index=False)
    
    print("Data cleaned and stored")


# -----------------------------
# Step 3: Data Warehouse Layer
# -----------------------------
def load_to_warehouse():
    print("Loading into Data Warehouse...")
    
    df = pd.read_csv('../processed_data/processed_faculty.csv')
    
    conn = sqlite3.connect('../processed_data/faculty_lakehouse.db')
    
    df.to_sql('faculty_workload', conn, if_exists='replace', index=False)
    
    conn.close()
    
    print("Data loaded into warehouse")

# -----------------------------
# Lakehouse Pipeline
# -----------------------------
def run_lakehouse_pipeline():
    print("Starting Lakehouse Pipeline...\n")
    
    load_to_data_lake()
    process_data()
    load_to_warehouse()
    
    print("\nLakehouse Pipeline Completed!")

# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    run_lakehouse_pipeline()

Starting Lakehouse Pipeline...

Loading data into Data Lake...
Data stored in Data Lake
Processing data...
Data cleaned and stored
Loading into Data Warehouse...
Data loaded into warehouse

Lakehouse Pipeline Completed!
